# Ray tracing and simplicial sheet fields

This compact example initializes a hydro grid and four-ray beam, traces the rays through a linear density gradient, and builds the two sheet-resolved tetrahedral fields.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from pyGATH.fields import simplicialise_sheet_fields
from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_simplicial_mesh

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
simulation = load_simulation_config(
    root / "configs/example_configs/tetrahedral_linear_gradient.toml"
)

In [ ]:
with simulation.reporting():
    grid = simulation.build_grid()
    beams = simulation.load_beams()
    initial_rays = simulation.initialize_rays(grid, beams=beams)
    trace = simulation.trace_rays(initial_rays, grid)
    field = simplicialise_sheet_fields(
        trace.sheet_fields,
        dimension=3,
        fields=("phase_length", "path_length", "inverse_brems_deposition"),
    )
print(
    f"{field.mesh.nsimplices:,} tetrahedra per sheet; terminated={bool(trace.terminated)}"
)

In [ ]:
figure = plt.figure(figsize=(11, 5))
for sheet in range(field.mesh.nsheets):
    axis = figure.add_subplot(1, field.mesh.nsheets, sheet + 1, projection="3d")
    plot_simplicial_mesh(field, sheet_index=sheet, ax=axis)
    axis.set_title(f"Sheet {sheet + 1}")
figure.tight_layout()